# Détection d'une figure : du simulateur à l'entraînement
Dataset synthétique → apprentissage → feature maps → prédiction + NMS.

In [ ]:
import os, random
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset

os.chdir(os.path.dirname(os.path.abspath("")) if os.path.basename(os.path.abspath("")) == "notebooks" else ".")

from src.dataset_generator import make_example
from src.detector import Detector, STRIDE, predict
from src.visualize import show_images, show_feature_maps, show_detections
from src.detector import code
from src.train import evaluate, train

MODEL_SIZE = 64          # taille d'entrée utilisée à l'entraînement
assert MODEL_SIZE % STRIDE == 0   # la grille de feature maps fait MODEL_SIZE // STRIDE

## 1. Génération du dataset
On génère directement les images à la taille attendue par le modèle.

In [ ]:
rng = random.Random(123)
pil_images = [make_example(MODEL_SIZE, "circle", rng)[0] for _ in range(10)]
show_images(pil_images, f"Images générées ({MODEL_SIZE}x{MODEL_SIZE})")

Les images sont déjà générées à la taille attendue par le détecteur (`MODEL_SIZE`×`MODEL_SIZE`, multiple du stride) : aucun redimensionnement, on les convertit simplement en tenseurs `(3, H, W)` normalisés dans [0, 1].

In [ ]:
def to_tensor(image):
    return torch.from_numpy(np.asarray(image.convert("RGB"), dtype=np.float32).copy()).permute(2, 0, 1) / 255.0

images = torch.stack([to_tensor(im) for im in pil_images])
print(images.shape)

## 2. Génération d'un jeu de données d'entraînement plus grand
On génère 200 images pour l'entraînement et 40 images indépendantes pour la validation. Les étiquettes (en pixels) sont encodées en cibles `(5, G, G)`.

In [ ]:

def make_dataset(count, seed):
    rng = random.Random(seed)
    dataset_images, targets = [], []
    for _ in range(count):
        image, label = make_example(MODEL_SIZE, "circle", rng)
        cx, cy, width, height = map(float, label.split())
        dataset_images.append(to_tensor(image))
        targets.append(code(cx, cy, width, height, size=MODEL_SIZE))
    return TensorDataset(torch.stack(dataset_images), torch.stack(targets))



## 3. Entraînement du modèle


In [ ]:
TRAIN_SAMPLES = 200
VAL_SAMPLES = 40
BATCH_SIZE = 64
EPOCHS = 50

import tempfile
from datetime import datetime
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

# Les logs sont écrits hors du dossier synchronisé ownCloud (TensorBoard/TensorFlow
# n'y accède pas correctement : "runs/demo is not a directory").
LOG_ROOT = Path(tempfile.gettempdir()) / "tb_runs" / "demo"
LOG_ROOT.mkdir(parents=True, exist_ok=True)
print("Logs TensorBoard :", LOG_ROOT)

torch.manual_seed(0)
train_set = make_dataset(TRAIN_SAMPLES, seed=7)
val_set = make_dataset(VAL_SAMPLES, seed=8)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)

model = Detector()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
run_dir = LOG_ROOT / datetime.now().strftime('%Y%m%d-%H%M%S')
with SummaryWriter(log_dir=str(run_dir)) as writer:
    for epoch in range(1, EPOCHS + 1):
        mean_loss = train(model, train_loader, optimizer)
        val_iou, val_accuracy = evaluate(model, val_loader, MODEL_SIZE)
        writer.add_scalar("Loss/train", mean_loss, epoch)
        writer.add_scalar("IoU/validation", val_iou, epoch)
        writer.add_scalar("Accuracy/validation_IoU_gt_0.5", val_accuracy, epoch)
        print(f"epoch {epoch:2d}  loss {mean_loss:.4f}  "
              f"val IoU {val_iou:.3f}  IoU>0.5 {val_accuracy:.0%}")

model.eval();

In [ ]:
# Affiche une commande adaptée au terminal pour ouvrir TensorBoard
import os, shlex, subprocess, sys
command = [sys.executable, '-m', 'tensorboard.main', '--logdir', str(LOG_ROOT), '--port', '6006']
if os.name == 'nt':
    print('PowerShell : & ' + subprocess.list2cmdline(command))
else:
    print(shlex.join(command))
print('Puis ouvrir http://localhost:6006')


## 4. Feature maps

In [ ]:
with torch.no_grad():
    maps = model.feature_maps(images)   # (N, 32, 8, 8)
print(maps.shape)
for i in range(3):
    show_feature_maps(images[i], maps[i], n_maps=8)
    
print(model.backbone[0].weight[3, 0, :, :])

## 5. Prédiction, décodage et NMS : une boîte par image

In [ ]:
with torch.no_grad():
    logits = model(images)
results = [predict(l, MODEL_SIZE) for l in logits]
show_detections(list(images), [b for b, _ in results], [s for _, s in results])